# 半全场扩展历史数据验证

本笔记本默认只读取本地 `.codex/soccer-predict` 的扩展数据集、固定赛季评估 artifact 和模型注册表。它复核 14 项赛事（含芬超）的数据血缘、动态行数、赛制/阶段/赛季状态、regular-only 训练边界、逐场评估指标和注册部署门禁。所有数字都从当前 artifact 计算，不固化旧导出的行数、哈希、命中率或 candidate/shadow 名单。CI 可设置 `SOCCER_PREDICT_NOTEBOOK_MODE=ci-smoke` 执行内置最小结构 fixture；该模式只证明笔记本可执行，明确不得作为模型证据。历史概率质量不是下注胜率或 ROI。

In [ ]:
from pathlib import Path
import json
import os
import sys
import pandas as pd
from IPython.display import Markdown, display

workspace_hint = os.environ.get('GITHUB_WORKSPACE')
repo_candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
if workspace_hint:
    repo_candidates.append(Path(workspace_hint).resolve())
repo = next((path for path in repo_candidates if (path / 'scripts').is_dir()), None)
if repo is None:
    raise RuntimeError('无法定位 Football-predictions 仓库根目录')
sys.path.insert(0, str(repo))
from scripts import history_importer, htft_holdout_evaluator, league_model_manager

notebook_mode = os.environ.get('SOCCER_PREDICT_NOTEBOOK_MODE', 'artifact')
ci_smoke = notebook_mode == 'ci-smoke'
if ci_smoke:
    fixture_path = repo / 'analysis' / 'fixtures' / 'htft_history_validation_smoke.json'
    fixture = json.loads(fixture_path.read_text(encoding='utf-8'))
    assert fixture['fixture_type'] == 'htft-notebook-ci-smoke/1.0.0'
    assert fixture['evidence_eligible'] is False
    dataset_dir = evaluation_path = model_dir = None
else:
    runtime = repo / '.codex' / 'soccer-predict'
    dataset_dir = runtime / 'datasets' / 'league-history-expanded'
    evaluation_path = runtime / 'evaluations' / 'htft-fixed-seasons.json'
    model_dir = runtime / 'models' / 'league-history-expanded'
    required = (dataset_dir / 'manifest.json', evaluation_path, model_dir / 'registry.json')
    for path in required:
        if not path.is_file():
            raise FileNotFoundError(f'缺少本地 artifact：{path}')

In [ ]:
if ci_smoke:
    manifest = fixture['manifest']
    evaluation = fixture['evaluation']
    registry = fixture['registry']
else:
    manifest = history_importer.validate_bundle(dataset_dir)
    evaluation = json.loads(evaluation_path.read_text(encoding='utf-8'))
    htft_holdout_evaluator.validate_evaluation(evaluation, dataset_dir=dataset_dir)
    registry = league_model_manager.load_registry(model_dir)

assert evaluation['dataset']['manifest_bundle_hash'] == manifest['bundle_hash']
assert registry['dataset_manifest_hash'] == manifest['bundle_hash']
assert registry['evaluation_hash'] == evaluation['evaluation_hash']
assert evaluation['promotion']['end_to_end_promotion_eligible'] is False
display(Markdown(
    f"**同源语义验证通过。** 数据 `{manifest['bundle_hash']}`；"
    f"评估 `{evaluation['evaluation_hash']}`；注册表 `{registry['registry_hash']}`。"
))

In [ ]:
expected_league_keys = {'finland_veikkausliiga'} if ci_smoke else {
    'afc_champions_league', 'brazil_serie_a', 'england_premier_league',
    'finland_veikkausliiga', 'france_ligue_1', 'germany_bundesliga',
    'italy_serie_a', 'japan_j1', 'korea_k_league_1',
    'norway_eliteserien', 'spain_la_liga', 'sweden_allsvenskan',
    'uefa_champions_league', 'usa_mls',
}
manifest_keys = {item['league_key'] for item in manifest['leagues']}
assert manifest_keys == expected_league_keys, (
    f'赛事集合不匹配：missing={sorted(expected_league_keys - manifest_keys)}, '
    f'extra={sorted(manifest_keys - expected_league_keys)}'
)

coverage_rows = []
for item in sorted(manifest['leagues'], key=lambda row: row['league_key']):
    regime_counts = item['competition_regimes']
    regular_rows = sum(by_regime.get('regular', 0) for by_regime in regime_counts.values())
    partial_rows = sum(
        count
        for by_status in item['season_statuses'].values()
        for status, count in by_status.items()
        if status.startswith('partial_as_of_')
    )
    coverage_rows.append({
        '赛事': item['league'], 'league_key': item['league_key'],
        'manifest行数': item['rows'], 'regular行数': regular_rows,
        '特殊regime行数': item['rows'] - regular_rows,
        '未完整赛季行数': partial_rows,
        'UTC起点': item['utc_date_start'], 'UTC终点': item['utc_date_end'],
    })
coverage = pd.DataFrame(coverage_rows)
assert int(coverage['manifest行数'].sum()) == sum(item['rows'] for item in manifest['leagues'])
display(coverage)
display(Markdown(f"**当前 manifest 总行数：{int(coverage['manifest行数'].sum()):,}；赛事数：{len(coverage)}。**"))

In [ ]:
cohort_rows = []
for item in manifest['leagues']:
    for season, count in item['seasons'].items():
        cohort_rows.append({
            'league_key': item['league_key'], 'season': int(season), 'rows': count,
            'formats': item['format_versions'].get(season, {}),
            'phases': item['phase_groups'].get(season, {}),
            'statuses': item['season_statuses'].get(season, {}),
            'regimes': item['competition_regimes'].get(season, {}),
        })
cohorts = pd.DataFrame(cohort_rows).sort_values(['league_key', 'season'])
assert all(row for row in cohorts['formats'])
assert all(row for row in cohorts['phases'])
assert all(row for row in cohorts['statuses'])
assert all(row for row in cohorts['regimes'])
display(cohorts)
display(Markdown(
    '**阶段口径：** format/phase/status/regime 全部保留用于审计与切片；'
    '这不表示当前 manager 为每个阶段拟合独立模型。'
))

In [ ]:
metric_rows = []
for league in evaluation['leagues']:
    for split in league['splits']:
        metrics = split['model_only']['overall']['metrics']
        baseline = split['league_empirical_frequency_baseline']['metrics']
        metric_rows.append({
            'league_key': league['league_key'], 'split': split['split_id'],
            'role': split['role'], 'season': split['test_season'],
            '样本': metrics['sample_count'], 'log_loss': metrics['nine_class_log_loss'],
            'Brier': metrics['nine_class_brier'],
            'Top1': metrics['top_one_accuracy'], 'Top2': metrics['top_two_accuracy'],
            '基线log_loss': baseline['nine_class_log_loss'],
            '基线Brier': baseline['nine_class_brier'],
            '排除测试行': split['excluded_test_match_count'],
        })
metrics = pd.DataFrame(metric_rows).sort_values(['league_key', 'season'])
assert int(metrics['样本'].sum()) == evaluation['summary']['all_splits']['model_only']['overall']['metrics']['sample_count']
display(metrics)
display(Markdown(
    '**评估口径：** 表中数字由逐场 forecast 重新校验后汇总；'
    'partial/shadow 行只用于研究，不能参与部署晋级。'
))

In [ ]:
registry_rows = []
for entry in sorted(registry['leagues'], key=lambda row: row['league_key']):
    policy = entry['competition_regime_policy']
    evidence = entry['league_pair_gate_evidence']
    assert policy['allowed_regimes'] == ['regular']
    assert policy['included_rows'] + policy['excluded_rows'] == policy['source_rows']
    assert entry['training_rows'] == policy['included_rows']
    assert entry['formal_htft_eligible'] is False
    assert evidence['dataset_manifest_hash'] == manifest['bundle_hash']
    assert evidence['evaluation_hash'] == evaluation['evaluation_hash']
    assert evidence['model_hash'] == entry['model_hash']
    assert evidence['league_key'] == entry['league_key']
    assert evidence['formal_htft_eligible'] is False
    assert evidence['production_confidence_eligible'] is False
    registry_rows.append({
        'league_key': entry['league_key'], '训练行': entry['training_rows'],
        '排除特殊regime': policy['excluded_rows'],
        'deployment_status': entry['deployment_status'],
        'pair阈值': evidence['threshold'],
        'eligible': evidence['eligible_sample_count'],
        'covered': evidence['covered_count'], 'hits': evidence['hit_count'],
        'regime_warning': evidence['regime_warning'],
        'formal_htft_eligible': entry['formal_htft_eligible'],
    })
registry_audit = pd.DataFrame(registry_rows)
assert set(registry_audit['league_key']) == expected_league_keys
display(registry_audit)

In [ ]:
partial_2026 = cohorts[cohorts['season'].eq(2026)].copy()
for statuses in partial_2026['statuses']:
    assert statuses and all(name.startswith('partial_as_of_') for name in statuses)
display(partial_2026[['league_key', 'season', 'rows', 'statuses', 'regimes', 'phases']])
display(Markdown(
    '**结论：** 2026 未完整赛季只作 research/shadow；HT/FT 全部保持非正式。'
    '精确行数、指标和 candidate/shadow 状态必须以本次已验证的 manifest、evaluation、registry 为准。'
))